## Modelo

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
)


In [107]:
DATA_PATH = "merged.csv"

data = pd.read_csv(DATA_PATH)
print(data.shape)
data.head()


(935557, 17)


,id_pedido,customer_id,pais,id_businessunit,business_unit,cedis,fecha_pedido,status_final,valor_pedido,SubTotal,Total,id_linea,sku_solicitado,nombre_sku_solicitado,quantity,status,fue_sustituida
0,8.839440e+18,8.429210e+18,México,1,Bebidas,3012,2025-03-17 14:00:00,Entregado,7,1366.8,1570.94,22789344,4120450000000000000,COCA - COLA,1,REGISTRADO,0
1,8.839440e+18,8.429210e+18,México,1,Bebidas,3012,2025-03-17 14:00:00,Entregado,7,1366.8,1570.94,22680333,6786130000000000000,FRUTSI FRUTAS ROJAS,1,ENTREGADO,0
2,8.839440e+18,8.429210e+18,México,1,Bebidas,3012,2025-03-17 14:00:00,Entregado,7,1366.8,1570.94,22789345,7316010000000000000,FANTA FRESA,1,REGISTRADO,0
3,8.839440e+18,8.429210e+18,México,1,Bebidas,3012,2025-03-17 14:00:00,Entregado,7,1366.8,1570.94,22680334,635733000000000000,FRUTSI UVA,1,ENTREGADO,0
4,8.839440e+18,8.429210e+18,México,1,Bebidas,3012,2025-03-17 14:00:00,Entregado,7,1366.8,1570.94,22789346,1254630000000000000,SIDRAL MUNDET MANZANA,1,REGISTRADO,0


In [108]:
# Validaciones basicas de la base antes de modelar.
print("Archivo:", DATA_PATH)
print("Shape inicial:", data.shape)
print("Filas duplicadas:", data.duplicated().sum())
print("id_linea duplicados:", data["id_linea"].duplicated().sum() if "id_linea" in data.columns else "sin id_linea")
print("Distribucion target:")
print(data["fue_sustituida"].value_counts(dropna=False))
print("Porcentaje target:")
print((data["fue_sustituida"].value_counts(normalize=True, dropna=False) * 100).round(4))


Archivo: merged.csv
Shape inicial: (935557, 17)
Filas duplicadas: 0
id_linea duplicados: 0
Distribucion target:
fue_sustituida
0    932232
1      3325
Name: count, dtype: int64
Porcentaje target:
fue_sustituida
0    99.6446
1     0.3554
Name: proportion, dtype: float64


In [95]:
# Capa de seguridad: el modelo debe tener una fila por id_linea.
# Si la base viene inflada por un merge many-to-many, esto evita que la misma linea
# caiga en train y test. Lo ideal sigue siendo corregir el merge desde database_clean.
data = data.drop_duplicates().copy()

if "id_linea" in data.columns:
    data = data.drop_duplicates(subset="id_linea", keep="first").copy()

print("Shape despues de deduplicar:", data.shape)
print("id_linea duplicados:", data["id_linea"].duplicated().sum())
print(data["fue_sustituida"].value_counts())


Shape despues de deduplicar: (935557, 17)
id_linea duplicados: 0
fue_sustituida
0    932232
1      3325
Name: count, dtype: int64


In [96]:
print("Cantidad de lineas sustituidas:", int(data["fue_sustituida"].sum()))
print("Tasa de sustitucion:", data["fue_sustituida"].mean())


Cantidad de lineas sustituidas: 3325
Tasa de sustitucion: 0.003554032517526992


In [97]:
target = "fue_sustituida"

# Columnas que no deben entrar al modelo.
# - IDs: identifican registros, pero no son patrones generalizables.
# - Columnas de cambio: revelan el resultado despues de ocurrida la sustitucion.
# - Status/fecha derivada: pueden ser informacion posterior o artificial en esta base.
cols_no_modelo = [
    "fue_sustituida",
    "id_linea",
    "id_pedido",
    "sku_solicitado_cambio",
    "nombre_sku_solicitado_cambio",
    "status",
    "status_final",
    "fecha_pedido",
    "hora",
    "dia_semana",
    "mes",
    "es_fin_semana",
]

cols_no_modelo = [c for c in cols_no_modelo if c in data.columns]

X = data.drop(columns=cols_no_modelo)
y = data[target].astype(int)
groups = data["id_pedido"]

print("Columnas usadas en X:", list(X.columns))
print("Columnas eliminadas:", cols_no_modelo)


Columnas usadas en X: ['customer_id', 'pais', 'id_businessunit', 'business_unit', 'cedis', 'valor_pedido', 'SubTotal', 'Total', 'sku_solicitado', 'nombre_sku_solicitado', 'quantity']
Columnas eliminadas: ['fue_sustituida', 'id_linea', 'id_pedido', 'status', 'status_final', 'fecha_pedido']


In [98]:
cat_cols = [
    "customer_id",
    "pais",
    "id_businessunit",
    "business_unit",
    "cedis",
    "sku_solicitado",
    "nombre_sku_solicitado",
]

cat_cols = [c for c in cat_cols if c in X.columns]

for col in cat_cols:
    X[col] = X[col].astype("string").fillna("missing")

num_cols = [c for c in X.columns if c not in cat_cols]

for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("Categoricas:", cat_cols)
print("Numericas:", num_cols)


Categoricas: ['customer_id', 'pais', 'id_businessunit', 'business_unit', 'cedis', 'sku_solicitado', 'nombre_sku_solicitado']
Numericas: ['valor_pedido', 'SubTotal', 'Total', 'quantity']


In [99]:
# Split por pedido: evita que lineas del mismo id_pedido caigan en train y test.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

pedidos_train = set(groups.iloc[train_idx])
pedidos_test = set(groups.iloc[test_idx])

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())
print("id_pedido en train y test:", len(pedidos_train & pedidos_test))


Train: (750389, 11) {0: 747770, 1: 2619}
Test: (185168, 11) {0: 184462, 1: 706}
id_pedido en train y test: 0


In [100]:
train_pool = Pool(
    X_train,
    y_train,
    cat_features=cat_cols,
)

test_pool = Pool(
    X_test,
    y_test,
    cat_features=cat_cols,
)


In [101]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
)

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True,
)


0:	learn: 0.9682425	test: 0.9745573	best: 0.9745573 (0)	total: 281ms	remaining: 2m 20s
100:	learn: 0.9992955	test: 0.9695147	best: 0.9749606 (2)	total: 22s	remaining: 1m 26s
200:	learn: 0.9995329	test: 0.9705385	best: 0.9749606 (2)	total: 46s	remaining: 1m 8s
300:	learn: 0.9996596	test: 0.9712091	best: 0.9749606 (2)	total: 1m 11s	remaining: 47.4s
400:	learn: 0.9997850	test: 0.9727659	best: 0.9749606 (2)	total: 1m 41s	remaining: 25s
499:	learn: 0.9998346	test: 0.9738897	best: 0.9749606 (2)	total: 2m 13s	remaining: 0us

bestTest = 0.9749605843
bestIteration = 2

Shrink model to first 3 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, eval_metric='PRAUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [102]:
y_proba = model.predict_proba(X_test)[:, 1]

# El threshold 0.5 se reporta solo como referencia.
# Para negocio conviene usar ranking/top-K.
y_pred_05 = (y_proba >= 0.5).astype(int)

print("PR-AUC:", average_precision_score(y_test, y_proba))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nMatriz de confusion con threshold 0.5:")
print(confusion_matrix(y_test, y_pred_05))

print("\nReporte con threshold 0.5:")
print(classification_report(y_test, y_pred_05))


PR-AUC: 0.8169885119197825
ROC-AUC: 0.9631943394807158

Matriz de confusion con threshold 0.5:
[[183995    467]
 [   118    588]]

Reporte con threshold 0.5:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    184462
           1       0.56      0.83      0.67       706

    accuracy                           1.00    185168
   macro avg       0.78      0.92      0.83    185168
weighted avg       1.00      1.00      1.00    185168



In [103]:
resultados_test = data.loc[X_test.index, [
    "id_pedido",
    "id_linea",
    "customer_id",
    "cedis",
    "sku_solicitado",
    "nombre_sku_solicitado",
    "quantity",
]].copy()

resultados_test["fue_sustituida_real"] = y_test.values
resultados_test["prob_sustitucion"] = y_proba

for pct in [0.01, 0.05, 0.10]:
    top_k = resultados_test.sort_values("prob_sustitucion", ascending=False).head(
        max(1, int(len(resultados_test) * pct))
    )
    precision_k = top_k["fue_sustituida_real"].mean()
    recall_k = top_k["fue_sustituida_real"].sum() / y_test.sum()
    lift_k = precision_k / y_test.mean()

    print(f"Top {pct:.0%}")
    print("Precision:", precision_k)
    print("Recall:", recall_k)
    print("Lift:", lift_k)
    print("Casos revisados:", len(top_k))
    print("Sustituciones encontradas:", int(top_k["fue_sustituida_real"].sum()))


Top 1%
Precision: 0.3192868719611021
Recall: 0.8371104815864022
Lift: 83.74180100183194
Casos revisados: 1851
Sustituciones encontradas: 591
Top 5%
Precision: 0.06470079930870598
Recall: 0.8484419263456091
Lift: 16.969571680445423
Casos revisados: 9258
Sustituciones encontradas: 599
Top 10%
Precision: 0.03310650248433787
Recall: 0.8682719546742209
Lift: 8.683094691246282
Casos revisados: 18516
Sustituciones encontradas: 613


In [104]:
importances = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance(),
}).sort_values("importance", ascending=False)

importances.head(20)


,feature,importance
9,nombre_sku_solicitado,73.877221
8,sku_solicitado,26.121617
10,quantity,0.001162
0,customer_id,0.000000
1,pais,0.000000
2,id_businessunit,0.000000
3,business_unit,0.000000
4,cedis,0.000000
5,valor_pedido,0.000000
6,SubTotal,0.000000


In [105]:
df_alertas = data.copy()

X_full = df_alertas[X.columns].copy()

for col in cat_cols:
    X_full[col] = X_full[col].astype("string").fillna("missing")

for col in [c for c in X_full.columns if c not in cat_cols]:
    X_full[col] = pd.to_numeric(X_full[col], errors="coerce")

df_alertas["prob_sustitucion"] = model.predict_proba(X_full)[:, 1]

df_alertas["nivel_riesgo"] = pd.qcut(
    df_alertas["prob_sustitucion"].rank(method="first"),
    q=[0, 0.90, 0.98, 1.0],
    labels=["bajo", "medio", "alto"],
)

df_alertas[[
    "id_pedido",
    "id_linea",
    "customer_id",
    "cedis",
    "sku_solicitado",
    "nombre_sku_solicitado",
    "quantity",
    "prob_sustitucion",
    "nivel_riesgo",
]].sort_values("prob_sustitucion", ascending=False).head(20)


,id_pedido,id_linea,customer_id,cedis,sku_solicitado,nombre_sku_solicitado,quantity,prob_sustitucion,nivel_riesgo
250279,9.075920e+18,11316769,5.098700e+18,3837,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
559490,9.082260e+18,19897869,5.233520e+18,3001,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,2,0.750228,alto
847104,8.984600e+18,1307225,6.075270e+18,3104,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
916820,8.930550e+18,11064639,3.314190e+18,3916,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
452650,8.870530e+18,4436427,1.299820e+18,3802,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
814153,9.223130e+18,20392507,2.868620e+18,3501,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
712726,8.876040e+18,411535,5.999520e+18,3308,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
250537,8.920050e+18,250570,2.910940e+18,3901,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
789927,8.889270e+18,20622840,2.913280e+18,3012,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
795393,9.091420e+18,12649631,5.966250e+18,3501,857838000000000000,COCA - COLA LIGHT SIN CAFEÍNA,1,0.750228,alto
